<a href="https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-1/08-cuando-el-85-no-sirve/clase.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Episodio 1.8 · Cuando el 85 % de aciertos no sirve

**Serie:** Del Excel al Machine Learning · **Temporada 1** · 🎬 Video próximamente

## 🎯 Qué vas a aprender

- Por qué la **exactitud** puede engañar.
- Leer una **matriz de confusión**, que no es más que una tabla dinámica.
- Calcular **precisión** y **exhaustividad**.
- Elegir el **umbral** según lo que cuesta equivocarse.

## 📊 En Excel lo harías así

Quieres comprobar un modelo. Pones en una tabla dinámica la columna **real** en las filas y la **predicha** en las columnas, y cuentas las filas. Con eso ya tienes casi todo lo de hoy.

Pero antes, un truco. Di que **ninguna** venta caerá en mora. ¿Cuánto aciertas?

## 🐍 En Python

### Paso 1: el modelo del episodio anterior

Estas celdas dejan listo el modelo del 1.7: `real` tiene lo que pasó y `probabilidad`, lo que estimó.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DATOS = "https://raw.githubusercontent.com/PercyTechX/del-excel-al-ml/main/datos/"

In [2]:
def preparar_x(tabla, numeros, textos, columnas=None):
    """Convierte los textos en columnas de 0 y 1. Con `columnas`, usa las de entrenamiento."""
    x = pd.get_dummies(tabla[numeros + textos], columns=textos, dtype=float)
    if columnas is not None:
        x = x.reindex(columns=columnas, fill_value=0)
    return x

In [3]:
def preparar_creditos(creditos):
    """Agrega las variables del modelo y separa entrenamiento y prueba al 1 de julio de 2025."""
    tabla = creditos.assign(
        log_monto=np.log(creditos["monto_sobre_limite"]),
        tasa_previa=(creditos["morosos_previos"] / creditos["casos_previos"]).fillna(0),
        sin_historial=(creditos["casos_previos"] == 0).astype(int),
    )
    fecha = pd.to_datetime(tabla["fecha"])
    se_sabe = fecha + pd.to_timedelta(tabla["plazo_dias"] + 31, unit="D")
    con_resultado = tabla["moroso"].notna()
    corte = pd.Timestamp("2025-07-01")
    return tabla[(se_sabe <= corte) & con_resultado], tabla[(fecha >= corte) & con_resultado]

In [4]:
creditos = pd.read_csv(DATOS + "creditos.csv")

entreno, prueba = preparar_creditos(creditos)

numeros = ["log_monto", "antiguedad_anios", "tasa_previa", "sin_historial"]
textos = ["tipo"]

x_entreno = preparar_x(entreno, numeros, textos)
x_prueba = preparar_x(prueba, numeros, textos, x_entreno.columns)

modelo = make_pipeline(StandardScaler(), LogisticRegression())
modelo.fit(x_entreno, entreno["moroso"])

probabilidad = modelo.predict_proba(x_prueba)[:, 1]

real = prueba["moroso"].astype(int).to_numpy()
tasa_previa = prueba["tasa_previa"].to_numpy()

### Paso 2: la trampa de la exactitud

La **exactitud** es el porcentaje de aciertos. `.mean()` de una comparación la calcula, igual que al contar `True` en el episodio 0.5.

In [5]:
print(round((real == 0).mean(), 3))

marcadas_05 = probabilidad >= 0.5
print(round(accuracy_score(real, marcadas_05), 3))
print(marcadas_05.sum())

0.854
0.853
6


**Decir que nadie cae en mora acierta el 85.4 %**, sin hacer nada. Nuestro modelo, con umbral 0.5, da 85.3 %: ¡un poco menos! Solo marca 6 compras. Es el truco: cuando un caso es raro, acertar es fácil, y la exactitud no dice nada.

### Paso 3: la matriz de confusión

Hay cuatro casos posibles. `pd.crosstab()` es la tabla dinámica que los cuenta. Probamos el umbral 0.25:

In [6]:
marcadas = probabilidad >= 0.25

pd.crosstab(real, marcadas, rownames=["Real"], colnames=["Marcada"])

Marcada,False,True
Real,,
0,1166,142
1,138,85


| | No marcada | Marcada |
|---|---|---|
| **Sin mora (0)** | Bien: no molestamos a nadie | **Falsa alarma** |
| **Con mora (1)** | **Se nos escapó** | Bien: la encontramos |

### Paso 4: dos preguntas mejores

- **Precisión:** de lo que marqué, ¿cuánto era mora de verdad? Mide las falsas alarmas.
- **Exhaustividad:** de toda la mora que hubo, ¿cuánta encontré? Mide lo que se escapó.

In [7]:
print(round(accuracy_score(real, marcadas), 3))
print(round(precision_score(real, marcadas), 3))
print(round(recall_score(real, marcadas), 3))

0.817
0.374
0.381


Bajó la exactitud, pero ahora encontramos el **38 %** de la mora. De cada 100 ventas marcadas, unas 37 sí eran mora. Es muchísimo más útil que el modelo "exacto" que no marca a nadie.

### Paso 5: ¿le gana a una regla?

Como siempre, una línea base: marcar a quien tenga una tasa de morosidad previa de 30 % o más.

In [8]:
regla = tasa_previa >= 0.3

print(regla.sum())
print(round(accuracy_score(real, regla), 3))
print(round(precision_score(real, regla), 3))
print(round(recall_score(real, regla), 3))

271
0.794
0.328
0.399


Marca más, con resultados parecidos: precisión de 33 % y exhaustividad de 40 %. **Una columna y un `=SI()` (`IF`) casi igualan al modelo.** Antes de implementar un modelo, pregúntate si una regla basta.

### Paso 6: el umbral según el costo

Estos datos no traen costos, y toda deuda termina pagándose. Así que **vamos a suponer** dos cifras, solo para el ejemplo: revisar una venta cuesta S/ 10, y dejar escapar una mora cuesta S/ 60. Esta función calcula el costo total de cada umbral:

In [9]:
def costo_total(real, probabilidad, umbral, costo_revisar, costo_escape):
    marcadas = probabilidad >= umbral
    revisadas = marcadas.sum()
    escapadas = (real == 1).sum() - real[marcadas].sum()
    return revisadas * costo_revisar + escapadas * costo_escape


for umbral in [0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5]:
    print(umbral, costo_total(real, probabilidad, umbral, 10, 60))

print("No revisar a nadie:", costo_total(real, probabilidad, 2, 10, 60))
print("Revisar todas:", costo_total(real, probabilidad, 0, 10, 60))
print("Regla del 30 %:", costo_total(real, tasa_previa, 0.3, 10, 60))

0.1 13430
0.15 11240
0.2 10470
0.25 10550
0.3 11380
0.4 12980
0.5 13320
No revisar a nadie: 13380
Revisar todas: 15310
Regla del 30 %: 10750


Con estos costos, el mejor umbral está alrededor de 0.2, lejos del 0.5 que suele usarse por defecto, que cuesta casi lo mismo que no revisar a nadie. Pero fíjate en la regla simple: cuesta apenas un poco más que el modelo. Si cambian los costos, cambia el umbral: lo verás en el ejercicio.

> ⚠️ Esto ilustra la idea, pero tiene supuestos fuertes: revisar evita todo el costo, y elegimos el umbral con los mismos datos de prueba. En serio, se elige con datos anteriores. Además, son datos sintéticos y una sola muestra: con otros datos simulados, cuántas marca el umbral 0.5 cambia mucho.

**Y antes de usarlo:** una marca significa que una persona revisa, no que se rechace a nadie. Una falsa alarma también le cuesta al cliente, y esos S/ 10 no lo incluyen. Y solo vemos a quienes ya recibieron crédito.

## 🤖 Pídeselo a la IA

> *"Un modelo detecta facturas con error. Solo el 2 % tiene errores y el modelo tiene 98 % de exactitud. Explícame por qué eso no me dice nada, y qué debería preguntar."*

La IA te ayuda a escribir y entender código, pero **tú debes entender lo que hace**. Revisa cada respuesta antes de usarla, y no pegues datos de tu empresa.

## ✅ Resumen

| En Excel | En Python |
|---|---|
| Tabla dinámica de real contra predicho | `pd.crosstab(real, marcadas)` |
| Porcentaje de aciertos | `accuracy_score(real, marcadas)` |
| De lo marcado, ¿cuánto era cierto? | `precision_score(real, marcadas)` |
| De lo cierto, ¿cuánto marqué? | `recall_score(real, marcadas)` |

## 📝 Ahora te toca

Abre [el ejercicio en Colab](https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-1/08-cuando-el-85-no-sirve/ejercicio.ipynb) y elige tu umbral. Cuando termines, compara con [la solución](https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-1/08-cuando-el-85-no-sirve/solucion.ipynb).

---

*Del Excel al Machine Learning* · por [Percy Nazario](https://www.linkedin.com/in/percy-nazario/) · [Repositorio](https://github.com/PercyTechX/del-excel-al-ml) · Contenido bajo licencia CC BY 4.0